# VCK190 Getting Started

This notebook demonstrates basic PYNQ functionality on the VCK190:
LEDs, buttons, DIP switches, DMA loopback, BRAM, and interrupts.

The base overlay uses **Versal segmented configuration**: the boot PDI
(golden reference) defines the PS/NoC/DDR contract, and the PL PDI
(`base.pdi`) is loaded at runtime without re-initializing the processor.

In [ ]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.pdi")
base.ip_dict

## LEDs

The `BaseOverlay` maps the `axi_gpio_led` IP to `base.leds[0..3]`.

In [ ]:
import time

# Turn on LED 0, wait, then turn it off
base.leds[0].on()
time.sleep(1)
base.leds[0].off()

# Toggle all LEDs
for i in range(4):
    base.leds[i].toggle()
time.sleep(0.5)
for i in range(4):
    base.leds[i].toggle()

## Push Buttons and DIP Switches

`base.buttons` maps to the user push buttons (active-high) and
`base.switches` maps to the 4-position DIP switch.

The VCK190 exposes **2 user push buttons** (GPIO_SW_N, GPIO_SW_S)
on the PL GPIO interface.

In [ ]:
n_buttons = 2
print(f"Number of buttons: {n_buttons}")
for i in range(n_buttons):
    print(f"  Button {i}: {base.buttons[i].read()}")

print(f"DIP switches: 0b{base.switches.read():04b}")

## DMA Loopback Test

The base design includes an AXI DMA engine whose MM2S output is looped
back to its S2MM input through an AXI Stream Data FIFO. This validates
the PL-to-DDR data path through the NoC.

In [ ]:
import numpy as np
from pynq import allocate

dma = base.dma

N = 1024
buf_send = allocate(shape=(N,), dtype=np.uint32)
buf_recv = allocate(shape=(N,), dtype=np.uint32)

print(f"Send buffer phys addr: {buf_send.device_address:#x}")
print(f"Recv buffer phys addr: {buf_recv.device_address:#x}")

buf_send[:] = np.arange(N, dtype=np.uint32)
buf_recv[:] = 0

dma.sendchannel.transfer(buf_send)
dma.recvchannel.transfer(buf_recv)
dma.sendchannel.wait()
dma.recvchannel.wait()

if np.array_equal(buf_send, buf_recv):
    print(f"DMA loopback PASSED ({N} x uint32 = {N*4} bytes)")
else:
    mismatches = np.where(buf_send != buf_recv)[0]
    print(f"DMA loopback FAILED — {len(mismatches)}/{N} words differ")
    print(f"  First 8 sent: {buf_send[:8]}")
    print(f"  First 8 recv: {buf_recv[:8]}")
    print(f"  First mismatch at index {mismatches[0]}: "
          f"sent={buf_send[mismatches[0]]:#x} recv={buf_recv[mismatches[0]]:#x}")

buf_send.freebuffer()
buf_recv.freebuffer()

## BRAM Read/Write Test

The base design includes 8 KB of block RAM at `axi_bram_ctrl_0`.

In [ ]:
bram = base.axi_bram_ctrl_0.mmio

bram.write(0x00, 0xDEADBEEF)
bram.write(0x04, 0xCAFEBABE)

val0 = bram.read(0x00)
val1 = bram.read(0x04)
print(f"BRAM[0x00] = 0x{val0:08X}  (expected 0xDEADBEEF)")
print(f"BRAM[0x04] = 0x{val1:08X}  (expected 0xCAFEBABE)")
assert val0 == 0xDEADBEEF
assert val1 == 0xCAFEBABE
print("BRAM test PASSED")


## Interrupts

Every interrupt in this design is routed through an AXI Interrupt Controller onto the
first PL-to-PS interrupt line, which is the arrangement PYNQ requires. The controller
tells PYNQ which source fired, so each IP's interrupt appears under its own driver.

`ip_dict` records what was found:

In [ ]:
for name, ip in sorted(base.ip_dict.items()):
    if ip["interrupts"]:
        for pin, details in ip["interrupts"].items():
            print(f"{name}/{pin}: controller={details['controller']} index={details['index']}")

### Waiting on a timer interrupt

The design includes two AXI Timers purely so an interrupt can be raised on demand,
without needing to press a button or start a transfer.

The sequence to fire one is: load a count into `TLR0`, pulse `LOAD0` to load it,
enable the interrupt output with `ENIT0`, set `UDT0` to count down, then start it with
`ENT0`. When the count reaches zero the timer raises its interrupt, and writing
`T0INT` clears it.

`interrupt.wait()` is a coroutine, so this cell awaits it directly. The timeout is
there so the cell cannot hang if the interrupt never arrives.

In [ ]:
import asyncio
import time

timer = base.axi_timer_0

async def wait_for_timer(cycles):
    timer.register_map.TLR0 = cycles
    timer.register_map.TCSR0.LOAD0 = 1
    timer.register_map.TCSR0.LOAD0 = 0
    timer.register_map.TCSR0.ENIT0 = 1
    timer.register_map.TCSR0.UDT0 = 1
    timer.register_map.TCSR0.ENT0 = 1
    await timer.interrupt.wait()
    timer.register_map.TCSR0.T0INT = 1

# The PL runs at 100 MHz, so this is about one second.
start = time.time()
await asyncio.wait_for(wait_for_timer(100_000_000), timeout=10)
print(f"interrupt arrived after {time.time() - start:.3f} s")